In [1]:
# ===========================================================
# CELDA 1 — Instalación con Playwright (más estable en Colab)
# ===========================================================
!pip install -q playwright nest_asyncio
!playwright install chromium
!playwright install-deps chromium
print("Playwright instalado")

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [2]:
import requests
import re
import time
import os
import logging
from datetime import datetime
from bs4 import BeautifulSoup
import chardet
import nest_asyncio

# Permite usar Playwright sync dentro del loop asyncio de Colab
nest_asyncio.apply()

# Logging con timestamps — mucho más útil que print() para depurar
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)

print('Librerías importadas correctamente')

Librerías importadas correctamente


In [3]:
from google.colab import drive

log.info('Conectando con Google Drive...')
drive.mount('/content/drive')
log.info('Google Drive montado correctamente.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cambia CARPETA_RAIZ_DRIVE si tu carpeta raíz tiene otro nombre.
# Para agregar una norma nueva, añade una entrada al diccionario
# siguiendo el mismo esquema.

CARPETA_RAIZ_DRIVE = "/content/drive/MyDrive/TG_Maestria"

# Parámetros de red
MAX_REINTENTOS   = 3      # Intentos por página antes de abandonar
PAUSA_ENTRE_PAGS = 0.8    # Segundos entre peticiones (respeta el servidor)
MAX_PAGINAS      = 80     # Máximo de páginas paginadas por norma
TIMEOUT          = 15     # Segundos de espera por respuesta HTTP

CONFIGURACION_NORMAS = {
    "Codigo_Civil": {
        "url_inicial": "http://www.secretariasenado.gov.co/senado/basedoc/codigo_civil.html",
        "rangos": [range(1, 87), range(1494, 2546)],
        "ruta_drive": f"{CARPETA_RAIZ_DRIVE}/01_Corpus_Raw/Codigo_Civil/codigo_civil_intro_rag.md",
    },
    "Codigo_Comercio": {
        "url_inicial": "http://www.secretariasenado.gov.co/senado/basedoc/codigo_comercio.html",
        "rangos": [],          # Lista vacía = extraer todos los artículos
        "ruta_drive": f"{CARPETA_RAIZ_DRIVE}/01_Corpus_Raw/Codigo_Comercio/codigo_comercio_rag.md",
    },
    "Ley_1480_Estatuto_Consumidor": {
        "url_inicial": "http://www.secretariasenado.gov.co/senado/basedoc/ley_1480_2011.html",
        "rangos": [range(1, 59)],
        "ruta_drive": f"{CARPETA_RAIZ_DRIVE}/01_Corpus_Raw/Ley_1480/ley_1480_rag.md",
    },
    "Ley_222_1995": {
        "url_inicial": "http://www.secretariasenado.gov.co/senado/basedoc/ley_0222_1995.html",
        "rangos": [range(1, 1001)],
        "ruta_drive": f"{CARPETA_RAIZ_DRIVE}/01_Corpus_Raw/Ley_222/ley_222_rag.md",
    },
}

print(f'Configuración cargada — {len(CONFIGURACION_NORMAS)} norma(s) registrada(s)')
for nombre in CONFIGURACION_NORMAS:
    cfg = CONFIGURACION_NORMAS[nombre]
    rangos_desc = 'todos los artículos' if not cfg['rangos'] else str([list(r) for r in cfg['rangos']])
    print(f'   • {nombre}: {rangos_desc}')

Configuración cargada — 4 norma(s) registrada(s)
   • Codigo_Civil: [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86], [1494, 1495, 1496, 1497, 1498, 1499, 1500, 1501, 1502, 1503, 1504, 1505, 1506, 1507, 1508, 1509, 1510, 1511, 1512, 1513, 1514, 1515, 1516, 1517, 1518, 1519, 1520, 1521, 1522, 1523, 1524, 1525, 1526, 1527, 1528, 1529, 1530, 1531, 1532, 1533, 1534, 1535, 1536, 1537, 1538, 1539, 1540, 1541, 1542, 1543, 1544, 1545, 1546, 1547, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1555, 1556, 1557, 1558, 1559, 1560, 1561, 1562, 1563, 1564, 1565, 1566, 1567, 1568, 1569, 1570, 1571, 1572, 1573, 1574, 1575, 1576, 1577, 1578, 1579, 1580, 1581, 1582, 1583, 1584, 1585, 1586, 1587, 1588, 1589, 1590, 1591, 1592,

In [5]:
# ===========================================================
# CELDA 5 — Funciones auxiliares (VERSIÓN PLAYWRIGHT)
# ===========================================================
from playwright.sync_api import sync_playwright
import re, os
from datetime import datetime
from bs4 import BeautifulSoup

def en_rango(num: int, rangos: list) -> bool:
    if not rangos:
        return True
    return any(num in r for r in rangos)


def obtener_urls(url_base: str, max_paginas: int = MAX_PAGINAS) -> list:
    base_dir, archivo = url_base.rsplit('/', 1)
    base_dir += '/'
    nombre_base = archivo.replace('.html', '')
    urls = [url_base]
    urls += [f"{base_dir}{nombre_base}_pr{i:03d}.html" for i in range(1, max_paginas + 1)]
    return urls


"""def extraer_articulos_de_html(html: str, rangos: list, acumulado: dict) -> int:
    soup = BeautifulSoup(html, 'html.parser')

    # Eliminar índice de navegación
    for tag_id in ['selector_aj']:
        tag = soup.find(id=tag_id)
        if tag:
            tag.decompose()

    patron = re.compile(
        r'^ART[IÍ]CULO\s+(\d+)\s*(?:o\.?|°|\.?)?\s*',
        re.IGNORECASE
    )

    nuevos = 0
    for tag in soup.find_all(['p', 'div', 'td', 'span', 'li']):
        # Solo el texto directo, sin descender en hijos
        texto = tag.get_text(separator=' ').strip()
        if len(texto) < 15:
            continue

        m = patron.match(texto)
        if m:
            num = int(m.group(1))
            if en_rango(num, rangos):
                if len(texto) > len(acumulado.get(num, '')):
                    acumulado[num] = texto
                    nuevos += 1

    return nuevos"""

def extraer_articulos_de_html(html: str, rangos: list, acumulado: dict) -> tuple:
    soup = BeautifulSoup(html, 'html.parser')

    # Eliminar índice de navegación
    for tag_id in ['selector_aj']:
        tag = soup.find(id=tag_id)
        if tag:
            tag.decompose()

    patron = re.compile(
        r'^ART[IÍ]CULO\s+(\d+)\s*(?:o\.?|°|\.?)?\s*',
        re.IGNORECASE
    )

    nuevos = 0
    total_vistos = 0 # <--- NUEVO CONTADOR

    for tag in soup.find_all(['p', 'div', 'td', 'span', 'li']):
        texto = tag.get_text(separator=' ').strip()
        if len(texto) < 15:
            continue

        m = patron.match(texto)
        if m:
            total_vistos += 1 # <--- CONTAMOS CUALQUIER ARTÍCULO QUE EXISTA
            num = int(m.group(1))
            if en_rango(num, rangos):
                if len(texto) > len(acumulado.get(num, '')):
                    acumulado[num] = texto
                    nuevos += 1

    return nuevos, total_vistos


def guardar_markdown(nombre: str, articulos: dict, ruta: str) -> None:
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(f'# {nombre} — Extracto para RAG\n')
        f.write(f'> Generado el {timestamp}\n\n')
        for num in sorted(articulos):
            f.write(f'## Artículo {num}\n\n')
            f.write(f'{articulos[num]}\n\n')
            f.write('---\n\n')
    log.info(f'Guardado: {ruta} ({len(articulos)} artículos)')


print('✅ Funciones auxiliares definidas')

✅ Funciones auxiliares definidas


<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_19216/50082097.py:34: SyntaxWarning: invalid escape sequence '\s'
  r'^ART[IÍ]CULO\s+(\d+)\s*(?:o\.?|°|\.?)?\s*',


In [6]:
# ===========================================================
# CELDA 6 — Función principal (VERSIÓN PLAYWRIGHT)
# ===========================================================

import asyncio # Importar asyncio para usar await asyncio.sleep
from playwright.async_api import async_playwright # Importar la versión asíncrona

async def procesar_norma(nombre: str, config: dict) -> dict:
    log.info(f'=== Iniciando: {nombre} ===')
    stats = {'paginas_ok': 0, 'paginas_error': 0, 'articulos_encontrados': 0}
    articulos = {}
    urls = obtener_urls(config['url_inicial'])

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        paginas_vacias = 0

        for url in urls:
            try:
                # Cargar página y esperar que el JS termine
                response = await page.goto(url, timeout=20000, wait_until='networkidle')

                # 404 o error → fin de paginación
                if response is None or response.status == 404:
                    log.info(f'Fin de paginación en: {url}')
                    break

                # Esperar que aparezca contenido de artículo (máx 8s)
                try:
                    await page.wait_for_selector(
                        "text=/ARTÍCULO|ARTICULO/",
                        timeout=8000
                    )
                except:
                    paginas_vacias += 1
                    if paginas_vacias >=8 :
                        log.info('8 páginas sin artículos — fin de contenido.')
                        break
                    continue

                """html = await page.content()
                nuevos = extraer_articulos_de_html(html, config['rangos'], articulos)
                stats['paginas_ok'] += 1
                log.info(f'[OK] {url.split("/")[-1]} — nuevos artículos: {nuevos}')

                if nuevos == 0:
                    paginas_vacias += 1
                else:
                    paginas_vacias = 0"""
                html = await page.content()

                # Desempaquetamos los dos valores
                nuevos, total_vistos = extraer_articulos_de_html(html, config['rangos'], articulos)
                stats['paginas_ok'] += 1
                log.info(f'[OK] {url.split("/")[-1]} — guardados: {nuevos} (vistos en pág: {total_vistos})')

                # Evaluamos en base a los artículos totales que existen en el HTML
                if total_vistos == 0:
                    paginas_vacias += 1
                else:
                    paginas_vacias = 0

                if paginas_vacias >= 8:
                    log.info('8 páginas sin artículos — fin de contenido.')
                    break

                await asyncio.sleep(PAUSA_ENTRE_PAGS) # Usar asyncio.sleep en async funciones

            except Exception as exc:
                stats['paginas_error'] += 1
                log.error(f'Error en {url}: {exc}')

        await browser.close()

    stats['articulos_encontrados'] = len(articulos)

    if articulos:
        guardar_markdown(nombre, articulos, config['ruta_drive'])
    else:
        log.warning(f'No se encontraron artículos para {nombre}. Revisa la URL y los rangos.')

    return stats


print('✅ Función principal con Playwright definida')


✅ Función principal con Playwright definida


In [7]:
import asyncio

# ===========================================================
# CELDA 7 — Ejecución principal
# ===========================================================

async def main_execution():
    inicio = datetime.now()
    reporte = {}

    for nombre, config in CONFIGURACION_NORMAS.items():
        reporte[nombre] = await procesar_norma(nombre, config)

    duracion = (datetime.now() - inicio).seconds

    print('\n' + '='*55)
    print('  RESUMEN FINAL')
    print('='*55)
    print(f'{"Norma":<35} {"Artículos":>10} {"Págs OK":>8} {"Errores":>8}')
    print('-'*55)
    for nombre, s in reporte.items():
        print(f'{nombre:<35} {s["articulos_encontrados"]:>10} {s["paginas_ok"]:>8} {s["paginas_error"]:>8}')
    print('-'*55)
    print(f'Tiempo total: {duracion}s')
    print(f'Archivos guardados en: {CARPETA_RAIZ_DRIVE}')
    print('='*55)

# Run the asynchronous main execution
await main_execution()


  RESUMEN FINAL
Norma                                Artículos  Págs OK  Errores
-------------------------------------------------------
Codigo_Civil                              1138       81        0
Codigo_Comercio                           2037       64        0
Ley_1480_Estatuto_Consumidor                58        3        0
Ley_222_1995                               247        6        0
-------------------------------------------------------
Tiempo total: 371s
Archivos guardados en: /content/drive/MyDrive/TG_Maestria
